In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
import json
import os

# 1. Veriyi yükleme ve hazırlama
def load_and_prepare_data(bob_folder, output_csv=None):
    # JSON dosyalarını listeleme
    json_files = [f for f in os.listdir(bob_folder) if f.endswith(".json")]
    
    # Tüm paketleri depolamak için boş bir liste
    all_packets = []
    
    # Her JSON dosyasını döngüye alıp veriyi birleştirme
    for file in json_files:
        with open(os.path.join(bob_folder, file), "r") as f:
            data = json.load(f)
            all_packets.extend(data["packets"])
    
    # JSON verisini Pandas DataFrame'e dönüştürme
    df = pd.DataFrame(all_packets)
    
    # İlgili sütunları seçme
    df = df[["hour", "minute", "second", "size", "source_port", "destination_port", "ttl", "protocol", "tcp_flags"]]
    
    # Dinamik olarak protokolleri One-Hot kodlama
    df = pd.get_dummies(df, columns=["protocol"], prefix="protocol")
    
    # TCP bayraklarını (SYN, ACK, FIN) sayısal değerlere dönüştürme
    df["tcp_flags"] = df["tcp_flags"].astype(str)
    df["SYN"] = df["tcp_flags"].apply(lambda x: int("SYN=true" in x))
    df["ACK"] = df["tcp_flags"].apply(lambda x: int("ACK=true" in x))
    df["FIN"] = df["tcp_flags"].apply(lambda x: int("FIN=true" in x))
    
    # Gereksiz sütunları kaldırma
    df.drop(columns=["tcp_flags"], inplace=True)
    
    # TTL sütununu medyan değer ile doldurma
    df['ttl'].fillna((df['ttl'].median()), inplace=True)
    
    # Veriyi normalizasyon öncesi kaydetme (isteğe bağlı)
    if output_csv:
        df.to_csv(output_csv, index=False)
    
    return df

# 2. Normal davranış analizörü
class NormalBehaviorAnalyzer:
    def __init__(self, model_path, data_df, sequence_length=50):
        try:
            self.model = load_model(model_path)
            self.use_model = True
            print("Model başarıyla yüklendi.")
        except Exception as e:
            print(f"Model yüklenirken bir hata oluştu: {str(e)}")
            print("Model olmadan istatistiksel analiz yapılacak.")
            self.use_model = False
        
        self.df = data_df
        self.sequence_length = sequence_length
        self.scaler = MinMaxScaler()
        
        # NaN kontrolü
        if self.df.isna().any().any():
            print("Uyarı: DataFrame'de NaN değerler var. Bunlar doldurulacak.")
            for col in self.df.columns:
                if self.df[col].dtype != 'object' and self.df[col].isna().any():
                    self.df[col].fillna(self.df[col].median(), inplace=True)
        
        # Normalizasyon
        self.df_scaled = self.scaler.fit_transform(self.df)
        self.column_names = self.df.columns
        
        # Ölçeklendirilmiş veriyi DataFrame'e dönüştürme
        self.df_scaled_df = pd.DataFrame(self.df_scaled, columns=self.column_names)
        
        # Zamanla ilgili sütunları gruplandırma
        self.time_columns = [col for col in self.column_names if col in ['hour', 'minute', 'second']]
        
        # Paket boyutu ve diğer sütunlar
        self.size_column = 'size'
        self.port_columns = [col for col in self.column_names if 'port' in col]
        self.protocol_columns = [col for col in self.column_names if 'protocol' in col]
        self.flag_columns = [col for col in self.column_names if col in ['SYN', 'ACK', 'FIN']]
        
    def create_sequences(self, data):
        sequences = []
        for i in range(len(data) - self.sequence_length):
            sequences.append(data[i : i + self.sequence_length])
        return np.array(sequences)
    
    def calculate_reconstruction_error(self):
        if not self.use_model:
            print("Model yüklü olmadığı için yeniden oluşturma hatası hesaplanamıyor.")
            return None, None
        
        # Dizileri oluştur
        lstm_input = self.create_sequences(self.df_scaled)
        
        # Modelin yeniden oluşturmasını al
        reconstructed = self.model.predict(lstm_input)
        
        # Yeniden oluşturma hatasını hesapla (MSE)
        mse = np.mean(np.power(lstm_input - reconstructed, 2), axis=(1, 2))
        
        # Eşik değeri belirle (örneğin 95. yüzdelik)
        threshold = np.percentile(mse, 95)
        
        print(f"Ortalama yeniden oluşturma hatası: {np.mean(mse):.6f}")
        print(f"95. yüzdelik eşik değeri: {threshold:.6f}")
        
        return mse, threshold
    
    def analyze_normal_patterns(self):
        # Her özellik için normal aralıkları belirle
        analysis_results = {}
        
        # 1. Zamanla ilgili analiz
        if self.time_columns:
            # Saatlik, dakikalık, saniyelik dağılımların analizini yapalım
            time_analysis = {}
            
            for col in self.time_columns:
                original_values = self.df[col].values
                
                # Saatlerin dağılımını al - histogram olarak
                if col == 'hour':
                    bins = 24
                else:
                    bins = 60
                    
                hist, bin_edges = np.histogram(original_values, bins=bins)
                
                # En sık kullanılan zamanları belirle (en yüksek frekanslı %20 bin)
                sorted_bins = sorted([(i, count) for i, count in enumerate(hist)], 
                                    key=lambda x: x[1], reverse=True)
                
                # Frekansların toplamının %80'ini oluşturan binleri al
                total_count = sum(hist)
                cumulative_count = 0
                normal_bins = []
                
                for bin_idx, count in sorted_bins:
                    cumulative_count += count
                    normal_bins.append(bin_idx)
                    if cumulative_count / total_count > 0.8:  # %80 eşiği
                        break
                
                # Normal aralıkları belirle
                normal_ranges = []
                if normal_bins:
                    # Bitişik bin gruplarını belirle
                    normal_bins.sort()
                    current_range = [normal_bins[0], normal_bins[0]]
                    
                    for i in range(1, len(normal_bins)):
                        if normal_bins[i] == current_range[1] + 1:
                            # Bitişik bin, mevcut aralığı genişlet
                            current_range[1] = normal_bins[i]
                        else:
                            # Yeni bir aralık başlat
                            normal_ranges.append(current_range)
                            current_range = [normal_bins[i], normal_bins[i]]
                    
                    # Son aralığı ekle
                    normal_ranges.append(current_range)
                
                # Bin aralıklarını gerçek değerlere dönüştür
                formatted_ranges = []
                for range_start, range_end in normal_ranges:
                    start_val = bin_edges[range_start]
                    end_val = bin_edges[range_end + 1]
                    
                    if col == 'hour':
                        formatted_ranges.append(f"{int(start_val):02d}:00-{int(end_val):02d}:00")
                    else:
                        formatted_ranges.append(f"{int(start_val)}-{int(end_val)}")
                
                time_analysis[col] = {
                    "normal_ranges": formatted_ranges,
                    "most_active_value": bin_edges[np.argmax(hist)]
                }
            
            analysis_results["time"] = time_analysis
        
        # 2. Paket boyutu analizi
        size_values = self.df[self.size_column].values
        q1, q3 = np.percentile(size_values, [25, 75])
        iqr = q3 - q1
        lower_bound = max(0, q1 - 1.5 * iqr)
        upper_bound = q3 + 1.5 * iqr
        
        analysis_results["packet_size"] = {
            "median": np.median(size_values),
            "normal_range": f"{int(lower_bound)}-{int(upper_bound)}",
            "most_common_range": f"{int(q1)}-{int(q3)}"
        }
        
        # 3. Port analizi
        port_analysis = {}
        for col in self.port_columns:
            port_values = self.df[col].values
            # En sık kullanılan portları bul
            value_counts = pd.Series(port_values).value_counts()
            top_ports = value_counts.head(5).index.tolist()
            # Normal portlar (kullanımın %80'ini oluşturan)
            cumulative = 0
            normal_ports = []
            for port, count in value_counts.items():
                normal_ports.append(port)
                cumulative += count
                if cumulative / len(port_values) > 0.8:
                    break
            
            port_analysis[col] = {
                "top_ports": top_ports,
                "normal_ports": normal_ports[:10]  # İlk 10 portu göster
            }
        
        analysis_results["ports"] = port_analysis
        
        # 4. Protokol analizi
        if self.protocol_columns:
            protocol_usage = {}
            for col in self.protocol_columns:
                protocol_name = col.replace("protocol_", "")
                usage_percent = self.df[col].mean() * 100
                protocol_usage[protocol_name] = f"{usage_percent:.2f}%"
            
            # Kullanım yüzdesine göre sırala
            sorted_protocols = sorted(protocol_usage.items(), 
                                     key=lambda x: float(x[1].replace("%", "")), 
                                     reverse=True)
            
            analysis_results["protocols"] = {
                "usage_percent": dict(sorted_protocols),
                "dominant_protocols": [p[0] for p in sorted_protocols[:3]]
            }
        
        # 5. TCP Flag analizi
        if self.flag_columns:
            flag_analysis = {}
            for col in self.flag_columns:
                usage_percent = self.df[col].mean() * 100
                flag_analysis[col] = f"{usage_percent:.2f}%"
            
            analysis_results["tcp_flags"] = flag_analysis
        
        return analysis_results
    
    def plot_normal_patterns(self):
        plt.figure(figsize=(18, 12))
        
        # 1. Zamansal dağılım
        if self.time_columns:
            for i, col in enumerate(self.time_columns):
                plt.subplot(3, 3, i+1)
                sns.histplot(self.df[col], kde=True)
                plt.title(f"Normal {col} dağılımı")
                plt.xlabel(col)
                plt.ylabel("Frekans")
        
        # 2. Paket boyutu dağılımı
        plt.subplot(3, 3, 4)
        sns.histplot(self.df[self.size_column], kde=True)
        plt.title("Normal paket boyutu dağılımı")
        plt.xlabel("Paket boyutu")
        plt.ylabel("Frekans")
        
        # 3. Kaynak port dağılımı
        if 'source_port' in self.port_columns:
            plt.subplot(3, 3, 5)
            top_source_ports = self.df['source_port'].value_counts().head(10)
            sns.barplot(x=top_source_ports.index, y=top_source_ports.values)
            plt.title("En sık kullanılan kaynak portlar")
            plt.xlabel("Port")
            plt.ylabel("Frekans")
            plt.xticks(rotation=45)
        
        # 4. Hedef port dağılımı
        if 'destination_port' in self.port_columns:
            plt.subplot(3, 3, 6)
            top_dest_ports = self.df['destination_port'].value_counts().head(10)
            sns.barplot(x=top_dest_ports.index, y=top_dest_ports.values)
            plt.title("En sık kullanılan hedef portlar")
            plt.xlabel("Port")
            plt.ylabel("Frekans")
            plt.xticks(rotation=45)
        
        # 5. Protokol kullanımı
        if self.protocol_columns:
            plt.subplot(3, 3, 7)
            protocol_usage = [self.df[col].mean() for col in self.protocol_columns]
            protocol_names = [col.replace("protocol_", "") for col in self.protocol_columns]
            sns.barplot(x=protocol_names, y=protocol_usage)
            plt.title("Protokol kullanım oranları")
            plt.xlabel("Protokol")
            plt.ylabel("Kullanım oranı")
            plt.xticks(rotation=45)
        
        # 6. TCP Flag kullanımı
        if self.flag_columns:
            plt.subplot(3, 3, 8)
            flag_usage = [self.df[col].mean() for col in self.flag_columns]
            sns.barplot(x=self.flag_columns, y=flag_usage)
            plt.title("TCP Flag kullanım oranları")
            plt.xlabel("Flag")
            plt.ylabel("Kullanım oranı")
        
        plt.tight_layout()
        plt.savefig("normal_patterns.png")
        plt.close()
        print("Grafik 'normal_patterns.png' olarak kaydedildi.")
        
    def generate_report(self):
        analysis = self.analyze_normal_patterns()
        
        # Model yüklüyse yeniden oluşturma hatasını hesapla
        if self.use_model:
            mse, threshold = self.calculate_reconstruction_error()
        else:
            mse, threshold = None, None
        
        print("\n=== Normal Davranış Aralıkları Raporu ===\n")
        
        # 1. Zamansal analiz
        if "time" in analysis:
            print("⏰ Zaman Analizi:")
            for time_unit, data in analysis["time"].items():
                normal_ranges = ", ".join(data["normal_ranges"])
                print(f"  • Normal {time_unit} aralıkları: {normal_ranges}")
                print(f"  • En aktif {time_unit}: {int(data['most_active_value'])}")
            print()
        
        # 2. Paket boyutu
        if "packet_size" in analysis:
            print("📦 Paket Boyutu Analizi:")
            print(f"  • Normal paket boyutu aralığı: {analysis['packet_size']['normal_range']} bayt")
            print(f"  • En yaygın paket boyutu aralığı: {analysis['packet_size']['most_common_range']} bayt")
            print(f"  • Medyan paket boyutu: {analysis['packet_size']['median']:.1f} bayt")
            print()
        
        # 3. Port analizi
        if "ports" in analysis:
            print("🔌 Port Analizi:")
            for port_type, data in analysis["ports"].items():
                print(f"  • En sık kullanılan {port_type}lar: {', '.join(map(str, data['top_ports']))}")
                print(f"  • Normal kabul edilen {port_type}lar: {', '.join(map(str, data['normal_ports']))}")
            print()
        
        # 4. Protokol analizi
        if "protocols" in analysis:
            print("🌐 Protokol Analizi:")
            print(f"  • Dominant protokoller: {', '.join(analysis['protocols']['dominant_protocols'])}")
            for protocol, usage in list(analysis["protocols"]["usage_percent"].items())[:5]:
                print(f"  • {protocol}: {usage}")
            print()
        
        # 5. TCP Flag analizi
        if "tcp_flags" in analysis:
            print("🚩 TCP Flag Analizi:")
            for flag, usage in analysis["tcp_flags"].items():
                print(f"  • {flag}: {usage}")
            print()
        
        if threshold:
            print(f"📊 Normal davranış için MSE eşik değeri: {threshold:.6f}")
            print("(Bu eşik değerinin üstündeki MSE değerleri anormal kabul edilebilir)")
        
        # Görselleri de oluştur
        self.plot_normal_patterns()
        
        return analysis

# Ana çalıştırma kodu
if __name__ == "__main__":
    # Kullanıcıdan veri klasörü yolunu al
    bob_folder = "C:/Users/edadd/Desktop/bitirme/data/"
    
    # Veriyi yükle ve hazırla
    print("\nVeri yükleniyor ve hazırlanıyor...")
    df = load_and_prepare_data(bob_folder)
    print(f"Toplam {len(df)} paket yüklendi.")
    
    # Kullanıcıdan model yolunu al (isteğe bağlı)
    use_model = input("\nModeli kullanmak istiyor musunuz? (e/h): ").lower() == 'e'
    
    if use_model:
        model_path = "C:/Users/edadd/Desktop/bitirme/model/"
        # Normal davranış analizörünü oluştur
        analyzer = NormalBehaviorAnalyzer(model_path, df)
    else:
        # Model olmadan analiz yap
        print("Model kullanılmadan istatistiksel analiz yapılacak.")
        analyzer = NormalBehaviorAnalyzer(None, df)
    
    # Raporu oluştur ve görselleri çiz
    analyzer.generate_report()


Veri yükleniyor ve hazırlanıyor...


C:\Users\edadd\AppData\Local\Temp\ipykernel_18944\1552177111.py:44: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['ttl'].fillna((df['ttl'].median()), inplace=True)


Toplam 33064 paket yüklendi.
Model yüklenirken bir hata oluştu: File format not supported: filepath=C:/Users/edadd/Desktop/bitirme/model/. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(C:/Users/edadd/Desktop/bitirme/model/, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).
Model olmadan istatistiksel analiz yapılacak.

=== Normal Davranış Aralıkları Raporu ===

⏰ Zaman Analizi:
  • Normal hour aralıkları: 13:00-13:00
  • En aktif hour: 13
  • Normal minute aralıkları: 14-14, 14-15, 15-16, 17-18, 18-19, 20-20, 20-21, 21-22
  • En aktif minute: 14
  • Normal second aralıkları: 0-2, 4-6, 7-8, 10-14, 15-24, 25-31, 32-33, 35-36, 39-57, 58-59
  • En aktif second: 40

📦 Paket Boyutu Analizi:
  • Normal paket boyutu 